# LEAP-O Extension Pipeline

**Inspired by:** [LEAP-O: Learning to Predict Dynamic Obstacles for Safe Trajectory Planning](https://par.nsf.gov/biblio/10670472)  
**Lab:** Intelligent Cyber-Physical Systems (iCPS) Laboratory — Prof. Truong Nghiem

This notebook reproduces the core LEAP-O workflow in Python:
1. **GP regression** for obstacle displacement prediction + uncertainty
2. **GRU-RNN** data-dependent mean functions (GP+RNN vs GP-only baseline)
3. **Uncertainty propagation** for future obstacle positions
4. **Receding-horizon safe planning** with confidence-based collision avoidance

**Future extensions:** multi-obstacle scenes, real sensor logs, Julia DDDynObs integration, learned social interactions.

In [ ]:
!pip install -q gpytorch torch scipy matplotlib numpy

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import gpytorch
from scipy.stats import norm
from dataclasses import dataclass

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'Using device: {DEVICE}')

## 1. Simulation Setup (Paper Scenarios 1 & 2)

In [ ]:
@dataclass
class SimConfig:
    dt: float = 0.1
    horizon: int = 8
    detection_range: float = 20.0
    obstacle_radius: float = 2.0
    robot_half: float = 1.25
    confidence: float = 0.95
    noise_std: float = 1.0
    n0: int = 6

CFG = SimConfig()

def scenario_circular(t, noise_std=1.0):
    wx, wy = np.random.randn(2) * noise_std
    vx = 5 * np.cos(np.pi * t / 4) + wx
    vy = 5 * np.sin(np.pi * t / 4) + wy
    return vx, vy

def scenario_sine(t, noise_std=1.0):
    wx, wy = np.random.randn(2) * noise_std
    vx = -5 + wx
    vy = 5 * np.sin(np.pi * t / 4) + wy
    return vx, vy

def simulate_obstacle(steps, scenario_fn, p0=(18.0, -10.0), dt=0.1):
    pos = [np.array(p0, dtype=float)]
    for k in range(1, steps):
        vx, vy = scenario_fn(k * dt)
        pos.append(pos[-1] + dt * np.array([vx, vy]))
    return np.asarray(pos)

def to_displacements(positions):
    return np.diff(positions, axis=0)

positions = simulate_obstacle(120, scenario_circular)
disps = to_displacements(positions)
plt.figure(figsize=(7, 5))
plt.plot(positions[:, 0], positions[:, 1], 'b-', label='Obstacle path')
plt.scatter(positions[0, 0], positions[0, 1], c='green', label='Start')
plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.title('Scenario 1: Circular obstacle motion')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## 2. GP Motion Predictor (Eq. 6–9)

In [ ]:
class DisplacementGP(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel()
        )

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x), self.covar_module(x)
        )

class GPDisplacementModel:
    def __init__(self, mean_fn=None):
        self.mean_fn = mean_fn
        self.model = None
        self.likelihood = None

    def fit(self, times, displacements, iters=80):
        x = torch.tensor(times, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(displacements, dtype=torch.float32)
        self.likelihood = gpytorch.likelihoods.GaussianLikelihood()
        self.model = DisplacementGP(x, y, self.likelihood)
        self.model.to(DEVICE); self.likelihood.to(DEVICE)
        x, y = x.to(DEVICE), y.to(DEVICE)
        self.model.set_train_data(x, y)
        self.model.train(); self.likelihood.train()
        opt = torch.optim.Adam(self.model.parameters(), lr=0.08)
        mll = gpytorch.mlls.ExactMarginalLogLikelihood(self.likelihood, self.model)
        for _ in range(iters):
            opt.zero_grad()
            out = self.model(x)
            loss = -mll(out, y)
            loss.backward(); opt.step()

    def predict(self, query_times):
        xq = torch.tensor(query_times, dtype=torch.float32).unsqueeze(-1).to(DEVICE)
        self.model.eval(); self.likelihood.eval()
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred = self.likelihood(self.model(xq))
        mu = pred.mean.cpu().numpy()
        var = pred.variance.cpu().numpy()
        if self.mean_fn is not None:
            mu = self.mean_fn(query_times, mu)
        return mu, var

## 3. RNN Data-Dependent Mean (Eq. 59–70)

In [ ]:
class MeanGRU(nn.Module):
    def __init__(self, hidden=16, depth=1):
        super().__init__()
        self.gru = nn.GRU(input_size=2, hidden_size=hidden, num_layers=depth, batch_first=True)
        self.head = nn.Linear(hidden, 2)

    def forward(self, gp_mean_seq):
        out, _ = self.gru(gp_mean_seq.unsqueeze(0))
        return self.head(out.squeeze(0))

class LeapOMeanAdapter:
    def __init__(self, n0=6, hidden=16):
        self.n0 = n0
        self.net = MeanGRU(hidden=hidden).to(DEVICE)
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=1e-3)
        self.loss_fn = nn.MSELoss()

    def train_on_history(self, gp_means, targets, epochs=40):
        if len(targets) < self.n0:
            return
        x = torch.tensor(gp_means, dtype=torch.float32).to(DEVICE)
        y = torch.tensor(targets, dtype=torch.float32).to(DEVICE)
        self.net.train()
        for _ in range(epochs):
            self.optimizer.zero_grad()
            pred = self.net(x)
            loss = self.loss_fn(pred, y)
            loss.backward(); self.optimizer.step()

    def refine(self, gp_mu_x, gp_mu_y):
        seq = np.stack([gp_mu_x, gp_mu_y], axis=1)
        if len(seq) < self.n0:
            return gp_mu_x, gp_mu_y
        self.net.eval()
        with torch.no_grad():
            refined = self.net(torch.tensor(seq, dtype=torch.float32).to(DEVICE))
        refined = refined.cpu().numpy()
        return refined[:, 0], refined[:, 1]

## 4. Uncertainty Propagation (Eq. 11–13)

In [ ]:
def propagate_uncertainty(p0, mu_dx, mu_dy, var_dx, var_dy, noise_var):
    H = len(mu_dx)
    mean = np.zeros((H + 1, 2)); var = np.zeros((H + 1, 2))
    mean[0] = p0; var[0] = 0.0
    for i in range(H):
        mean[i + 1] = mean[i] + np.array([mu_dx[i], mu_dy[i]])
        var[i + 1] = var[i] + np.array([var_dx[i], var_dy[i]]) + noise_var
    return mean[1:], var[1:]

def confidence_margin(var_x, var_y, confidence=0.95):
    z = norm.ppf(0.5 + confidence / 2)
    return z * np.sqrt(var_x + var_y)

## 5. Receding-Horizon Safe Planner (Simplified NLP 19)

In [ ]:
def plan_safe_path(robot_p, goal_p, obs_mean, obs_var, cfg: SimConfig):
    H = min(cfg.horizon, len(obs_mean))
    path = [robot_p.copy()]
    p = robot_p.copy()
    for i in range(H):
        margin = confidence_margin(obs_var[i, 0], obs_var[i, 1], cfg.confidence)
        safe_radius = cfg.obstacle_radius + margin + cfg.robot_half
        to_goal = goal_p - p
        if np.linalg.norm(to_goal) < 1e-3:
            break
        step = 0.8 * to_goal / (np.linalg.norm(to_goal) + 1e-6)
        candidate = p + step
        if np.linalg.norm(candidate - obs_mean[i]) < safe_radius:
            lateral = np.array([-step[1], step[0]])
            candidate = p + 0.6 * lateral / (np.linalg.norm(lateral) + 1e-6)
        p = candidate
        path.append(p.copy())
    return np.asarray(path)

## 6. LEAP-O Online Loop (Algorithm 1)

In [ ]:
def run_leap_o(positions, t_eval, use_rnn=True, cfg: SimConfig = CFG):
    disps = to_displacements(positions)
    times = np.arange(len(disps)) * cfg.dt
    idx = np.where(times <= t_eval)[0]
    if len(idx) < 3:
        return None
    train_t = times[idx]
    train_dx = disps[idx, 0]; train_dy = disps[idx, 1]
    fut_t = train_t[-1] + cfg.dt * np.arange(1, cfg.horizon + 1)

    gp_x = GPDisplacementModel(); gp_x.fit(train_t, train_dx)
    gp_y = GPDisplacementModel(); gp_y.fit(train_t, train_dy)
    mu_dx, var_dx = gp_x.predict(fut_t)
    mu_dy, var_dy = gp_y.predict(fut_t)

    if use_rnn:
        rnn = LeapOMeanAdapter(n0=cfg.n0)
        hist_mu = np.stack([train_dx, train_dy], axis=1)
        rnn.train_on_history(hist_mu, hist_mu)
        mu_dx, mu_dy = rnn.refine(mu_dx, mu_dy)

    p0 = positions[idx[-1] + 1]
    noise_var = np.array([cfg.noise_std ** 2, cfg.noise_std ** 2])
    obs_mean, obs_var = propagate_uncertainty(p0, mu_dx, mu_dy, var_dx, var_dy, noise_var)

    gt = positions[idx[-1] + 1: idx[-1] + 1 + cfg.horizon]
    rmse = np.sqrt(np.mean(np.sum((gt - obs_mean[:len(gt)]) ** 2, axis=1))) if len(gt) else np.nan
    return dict(obs_mean=obs_mean, obs_var=obs_var, rmse=rmse, p0=p0, gt=gt)

eval_steps = [10, 20, 30, 40, 50]
rows = []
for t in eval_steps:
    gp_only = run_leap_o(positions, t * CFG.dt, use_rnn=False)
    gp_rnn = run_leap_o(positions, t * CFG.dt, use_rnn=True)
    if gp_only and gp_rnn:
        rows.append((t, gp_only['rmse'], gp_rnn['rmse']))

print('RMSE comparison (m) — GP-only vs GP+RNN (LEAP-O):')
print('Time | GP-only | GP+RNN')
for t, e1, e2 in rows:
    print(f'{t:4d} | {e1:7.2f} | {e2:7.2f}')

## 7. Visualization: Prediction + Safe Trajectory

In [ ]:
t_show = 20
result = run_leap_o(positions, t_show * CFG.dt, use_rnn=True)
robot_p = np.array([0.0, 2.0])
goal_p = np.array([robot_p[0] + 8.0, 2.0])
planned = plan_safe_path(robot_p, goal_p, result['obs_mean'], result['obs_var'], CFG)

plt.figure(figsize=(9, 6))
plt.plot(positions[:, 0], positions[:, 1], 'b-', alpha=0.4, label='Obstacle history')
plt.plot(result['obs_mean'][:, 0], result['obs_mean'][:, 1], 'c--', linewidth=2, label='Predicted obstacle')
for i, (m, v) in enumerate(zip(result['obs_mean'], result['obs_var'])):
    r = CFG.obstacle_radius + confidence_margin(v[0], v[1], CFG.confidence)
    circle = plt.Circle(m, r, color='orange', alpha=0.15)
    plt.gca().add_patch(circle)
plt.plot(planned[:, 0], planned[:, 1], 'g-o', markersize=4, label='Safe robot plan')
plt.axhline(2.0, color='brown', linestyle='--', alpha=0.5, label='Reference path')
plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.title(f'LEAP-O style planning at t={t_show}s')
plt.legend(); plt.grid(True, alpha=0.3); plt.axis('equal'); plt.show()

## 8. Future Extension Hooks

| Extension | How to extend in this notebook |
|---|---|
| Multi-obstacle (Scenario 3) | Run `run_leap_o` per obstacle; merge chance constraints in `plan_safe_path` |
| Real-world logs | Replace `simulate_obstacle` with CSV detections inside `Rd` |
| Full NLP solver | Replace heuristic planner with `cvxpy` / `casadi` implementation of (19) |
| Julia parity | Compare against [DDDynObs](https://github.com/EasternBoy/DDDynObs) reference code |
| Abrupt motion patterns | Add jerk/acceleration change scenarios mentioned in paper conclusions |

**Download:** `File → Download → Download .ipynb` then upload to Google Colab.